# HydroFlow V2 object detection

Run this notebook in Google Colab on a GPU (T4 is enough). It installs Ultralytics, builds a YOLO dataset whose class names are the HydroFlow inventory item codes, trains one YOLO11s detector from the COCO pretrained weights, and saves `best.pt` to `ml/models/object_detect/v2/best.pt`.

Do not run the download or training cells on the HydroFlow machine. Training stays in your Colab account. The app only loads the exported file. Until that file is present, the app keeps the three YOLOv8n category weights.

Rice bags, blankets, towels, mosquito nets, and men/women cloth need local photos. SKU-110K and DeepFashion2 are not the training set.


## Class map

`data.yaml` names match `ITEM_TYPES`, in this order: `tshirt`, `m_cloth`, `f_cloth`, `rice`, `oil`, `canned_fish`, `blanket`, `soap`, `cup`, `spoon`, `mug`, `box`, `towel`, `plate`, `netbed`, `knife`, `bowl`, `pot`.

### Open Images V7

Download a capped subset with FiftyOne. The classes are Bottle, Bowl, Box, Clothing, Dress, Coat, Knife, Spoon, and Coffee cup. License: CC BY 2.0.

| Open Images V7 class | Inventory code |
| --- | --- |
| Bottle | `oil` |
| Bowl | `bowl` |
| Box | `box` |
| Clothing | `tshirt` |
| Dress | `f_cloth` (weak prior only) |
| Coat | `m_cloth` (weak prior only) |
| Knife | `knife` |
| Spoon | `spoon` |
| Coffee cup | `cup` |

Clothing, Dress, and Coat are cloth. They do not name men cloth or women cloth. Dress mapped to `f_cloth` and Coat mapped to `m_cloth` is only a starting prior.

COCO 2017 is already inside `yolo11s.pt` (bottle, cup, bowl, knife, spoon). This notebook does not download COCO again.

### RPC

Use the canned-food, drink, and personal-hygiene meta-categories. Academic use fits CC BY-NC-SA 4.0. RPC is not a commercial training set.

| RPC meta-category | Inventory code |
| --- | --- |
| canned-food | `canned_fish` |
| drink | `oil` |
| personal-hygiene | `soap` |

### Local photos and the cap

Rice bags, blankets, towels, mosquito nets, and men/women cloth need local photos. Mug, plate, and pot are not in the lists above, so they need local photos too.

Use about 1,000–1,500 images per class. This notebook caps each class at 1,200. Hold out an entire photo session (one camera, one day) as the test set. A class with fewer than 200 boxes stays manual rather than being forced into a score.

SKU-110K and DeepFashion2 are not the training set. This notebook does not download them.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics>=8.3.0",
        "albumentations>=1.4.0",
        "fiftyone>=1.0.0",
        "pyyaml",
        "Pillow",
    ]
)
print("Installed ultralytics, albumentations, fiftyone, and pyyaml.")


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive was not mounted:", exc)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No nvidia-smi on this runtime. Training will use CPU if you continue.")

WORK = Path("/content/hydroflow_detect")
STAGED = WORK / "staged"
DRIVE = Path("/content/drive/MyDrive")
RPC_ROOT = DRIVE / "datasets" / "RPC"
LOCAL_ROOT = DRIVE / "HydroFlow_detect_local"
RUNS = WORK / "runs"
for path in (WORK, STAGED, RUNS):
    path.mkdir(parents=True, exist_ok=True)

# Inventory ITEM_TYPES order. These strings are the YOLO class names.
CLASS_NAMES = (
    "tshirt",
    "m_cloth",
    "f_cloth",
    "rice",
    "oil",
    "canned_fish",
    "blanket",
    "soap",
    "cup",
    "spoon",
    "mug",
    "box",
    "towel",
    "plate",
    "netbed",
    "knife",
    "bowl",
    "pot",
)
# About 1,000-1,500 images per class. 1,200 is the cap used in this notebook.
IMAGES_PER_CLASS_CAP = 1200
MIN_BOXES_BEFORE_MANUAL = 200

OI_TO_ITEM = {
    "Bottle": "oil",
    "Bowl": "bowl",
    "Box": "box",
    "Clothing": "tshirt",
    "Dress": "f_cloth",
    "Coat": "m_cloth",
    "Knife": "knife",
    "Spoon": "spoon",
    "Coffee cup": "cup",
}
RPC_META_TO_ITEM = {
    "canned-food": "canned_fish",
    "drink": "oil",
    "personal-hygiene": "soap",
}
LOCAL_REQUIRED = ("rice", "blanket", "towel", "netbed", "m_cloth", "f_cloth")
LOCAL_NOTE = (
    "Rice bags, blankets, towels, mosquito nets, and men/women cloth need local photos. "
    "SKU-110K and DeepFashion2 are not the training set."
)

assert set(OI_TO_ITEM.values()) <= set(CLASS_NAMES)
assert set(RPC_META_TO_ITEM.values()) <= set(CLASS_NAMES)
assert IMAGES_PER_CLASS_CAP in range(1000, 1501)
print(LOCAL_NOTE)
print("Class cap:", IMAGES_PER_CLASS_CAP)
for index, name in enumerate(CLASS_NAMES):
    print(f"{index:2d} {name}")


In [ ]:
import hashlib
import random
import shutil
from pathlib import Path

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RECORDS = []


def assert_allowed(name: str) -> None:
    lowered = name.lower().replace("_", "").replace("-", "")
    for banned in ("sku110k", "deepfashion", "deepfashion2"):
        if banned in lowered:
            raise RuntimeError(
                "SKU-110K and DeepFashion2 are not the training set and are not downloaded."
            )


def yolo_box(bbox, width=None, height=None):
    x, y, w, h = (float(value) for value in bbox)
    if max(x, y, w, h) > 1 and width and height:
        x, w = x / float(width), w / float(width)
        y, h = y / float(height), h / float(height)
    cx = min(1.0, max(0.0, x + (w / 2)))
    cy = min(1.0, max(0.0, y + (h / 2)))
    return cx, cy, min(1.0, max(0.0, w)), min(1.0, max(0.0, h))


def stage_image(source: Path, key: str) -> Path | None:
    if not source.is_file():
        return None
    dest = STAGED / f"{key}{source.suffix.lower()}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        shutil.copy2(source, dest)
    return dest


def add_record(key, source, origin, boxes, split_lock=None):
    assert_allowed(str(source))
    staged = stage_image(Path(source), key)
    if staged is None or not boxes:
        return
    RECORDS.append(
        {
            "key": key,
            "path": staged,
            "origin": origin,
            "split_lock": split_lock,
            "boxes": boxes,
        }
    )


def class_index(name: str) -> int:
    return CLASS_NAMES.index(name)


print("Helpers ready.", LOCAL_NOTE)


## Open Images V7

Each listed class is downloaded separately and capped at 1,200 images. Boxes are remapped to the inventory codes before the files are staged. Dress and Coat stay weak priors for women cloth and men cloth. Local session photos are still required for that split.


In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

assert_allowed("open-images-v7")


def detection_field(dataset):
    schema = dataset.get_field_schema()
    for name in ("ground_truth", "detections"):
        if name in schema:
            return name
    raise RuntimeError("Open Images samples have no detection field.")


def export_open_images(dataset, oi_name, item_code):
    field = detection_field(dataset)
    kept = 0
    for sample in dataset:
        detections = sample[field]
        if detections is None:
            continue
        width = getattr(getattr(sample, "metadata", None), "width", None)
        height = getattr(getattr(sample, "metadata", None), "height", None)
        boxes = []
        for det in detections.detections:
            if det.label != oi_name:
                continue
            cx, cy, w, h = yolo_box(det.bounding_box, width, height)
            if w <= 0 or h <= 0:
                continue
            boxes.append((item_code, cx, cy, w, h, oi_name))
        if not boxes:
            continue
        add_record(f"oi-{sample.id}", sample.filepath, "public", boxes)
        kept += 1
    return kept


for oi_name, item_code in OI_TO_ITEM.items():
    dataset_name = "hf_oi_" + item_code + "_" + str(abs(hash(oi_name)) % 10000)
    if fo.dataset_exists(dataset_name):
        fo.delete_dataset(dataset_name)
    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["detections"],
        classes=[oi_name],
        max_samples=IMAGES_PER_CLASS_CAP,
        shuffle=True,
        seed=17,
        dataset_name=dataset_name,
    )
    kept = export_open_images(dataset, oi_name, item_code)
    fo.delete_dataset(dataset_name)
    print(f"Open Images {oi_name} -> {item_code}: staged {kept} images (cap {IMAGES_PER_CLASS_CAP})")

print("Open Images subset staged. COCO was not downloaded.")


## RPC canned-food, drink, and hygiene

Place the official RPC archive at `/content/drive/MyDrive/datasets/RPC` (see [rpc-dataset.github.io](https://rpc-dataset.github.io/)). This cell reads COCO-style annotation JSON and keeps three meta-categories only:

- `canned-food` -> `canned_fish`
- `drink` -> `oil`
- `personal-hygiene` -> `soap`

`drinks` is accepted as `drink`. Hyphens, underscores, and spaces are the same character. If the archive is absent, training continues with Open Images and local photos. SKU-110K and DeepFashion2 are not read and not downloaded.


In [ ]:
def normalize_meta(value: str) -> str:
    text = value.strip().lower().replace("_", "-").replace(" ", "-")
    while "--" in text:
        text = text.replace("--", "-")
    if text == "drinks":
        return "drink"
    return text


def load_rpc(root: Path) -> None:
    assert_allowed(str(root))
    if not root.is_dir():
        print("RPC archive not found at", root)
        print("Skip RPC, or copy the official academic archive there and rerun this cell.")
        return
    json_files = [path for path in root.rglob("*.json") if path.is_file()]
    named = [
        path
        for path in json_files
        if "instance" in path.name.lower() or "annot" in path.name.lower()
    ]
    chosen = named or sorted(json_files, key=lambda path: path.stat().st_size, reverse=True)[:3]
    if not chosen:
        print("RPC root has no annotation JSON:", root)
        return
    image_index = {
        path.name: path
        for path in root.rglob("*")
        if path.suffix.lower() in IMAGE_SUFFIXES
    }
    added = 0
    seen_meta = set()
    for annotation_path in chosen:
        payload = json.loads(annotation_path.read_text(encoding="utf-8"))
        if "images" not in payload or "annotations" not in payload:
            continue
        categories = {
            int(row["id"]): row for row in payload.get("categories", []) if "id" in row
        }
        images = {int(row["id"]): row for row in payload["images"]}
        grouped = {}
        for ann in payload["annotations"]:
            grouped.setdefault(int(ann["image_id"]), []).append(ann)
        for image_id, anns in grouped.items():
            image = images.get(image_id)
            if image is None:
                continue
            file_name = str(image.get("file_name") or image.get("file_name_abs") or "")
            source = image_index.get(Path(file_name).name)
            if source is None:
                continue
            width = image.get("width")
            height = image.get("height")
            boxes = []
            for ann in anns:
                category = categories.get(int(ann.get("category_id", -1)), {})
                meta = normalize_meta(
                    str(category.get("supercategory") or category.get("name") or "")
                )
                seen_meta.add(meta)
                item_code = RPC_META_TO_ITEM.get(meta)
                if item_code is None:
                    continue
                bbox = ann.get("bbox")
                if not bbox or len(bbox) < 4:
                    continue
                cx, cy, w, h = yolo_box(bbox, width, height)
                if w <= 0 or h <= 0:
                    continue
                boxes.append((item_code, cx, cy, w, h, meta))
            if boxes:
                add_record(f"rpc-{annotation_path.stem}-{image_id}", source, "public", boxes)
                added += 1
    print("RPC meta-categories seen:", ", ".join(sorted(seen_meta)) or "(none)")
    print("RPC mapped images staged:", added)
    print("Mapped metas:", ", ".join(f"{key}->{value}" for key, value in RPC_META_TO_ITEM.items()))


load_rpc(RPC_ROOT)


## Local HydroFlow photos

Put warehouse photos on Drive like this:

```text
MyDrive/HydroFlow_detect_local/pool/<item_code>/*.jpg
MyDrive/HydroFlow_detect_local/sessions/<camera-day>/<item_code>/*.jpg
```

`pool` may enter train and val. Every `sessions/<camera-day>` folder is held out together as the test set, so the score is a new room rather than a random crop of the same pile.

A sibling `.txt` file may contain YOLO rows (`class cx cy w h`) or a class name in the first column. An image with no label file becomes one full-frame box of its folder class. Piles of mixed items need real boxes.

Rice bags, blankets, towels, mosquito nets, and men/women cloth need local photos.


In [ ]:
def parse_label_file(path: Path, fallback: str):
    if not path.is_file():
        return [(fallback, 0.5, 0.5, 1.0, 1.0, "full-frame")]
    boxes = []
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        head, cx, cy, w, h = parts
        if head.isdigit():
            name = CLASS_NAMES[int(head)]
        else:
            name = head.strip().lower()
        if name not in CLASS_NAMES:
            continue
        boxes.append((name, float(cx), float(cy), float(w), float(h), "local-label"))
    if boxes:
        return boxes
    return [(fallback, 0.5, 0.5, 1.0, 1.0, "full-frame")]


def ingest_class_dir(folder: Path, split_lock=None):
    if not folder.is_dir():
        return 0
    count = 0
    full_frame = 0
    for class_name in CLASS_NAMES:
        class_dir = folder / class_name
        if not class_dir.is_dir():
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            boxes = parse_label_file(image_path.with_suffix(".txt"), class_name)
            if any(box[5] == "full-frame" for box in boxes):
                full_frame += 1
            key = "local-" + hashlib.sha1(str(image_path).encode()).hexdigest()[:16]
            add_record(key, image_path, "local", boxes, split_lock=split_lock)
            count += 1
    if full_frame:
        print(folder, "used", full_frame, "full-frame labels. Mixed piles need real boxes.")
    return count


local_count = 0
if LOCAL_ROOT.is_dir():
    local_count += ingest_class_dir(LOCAL_ROOT / "pool")
    sessions = LOCAL_ROOT / "sessions"
    if sessions.is_dir():
        for session_dir in sorted(path for path in sessions.iterdir() if path.is_dir()):
            held = ingest_class_dir(session_dir, split_lock="test")
            local_count += held
            print("Held out session", session_dir.name, "images", held)
else:
    print("No local photo folder at", LOCAL_ROOT)

print("Local images staged:", local_count)
print(LOCAL_NOTE)
for name in LOCAL_REQUIRED:
    print("Local photos required for", name)


## Build `data.yaml`

Train/val is a 90/10 split of the capped pool. Held-out sessions stay in `test` and are not mixed back into train. The class list always has all 18 inventory codes, even when a class still needs photos.

Phase 1 trains on the public pool only. Phase 2 trains on public plus local pool images, and validates metrics on the held-out sessions when those exist.


In [ ]:
import yaml


def assign_split(record) -> str:
    if record["split_lock"] == "test":
        return "test"
    digest = hashlib.sha256(record["key"].encode("utf-8")).hexdigest()
    return "val" if int(digest[:2], 16) < 26 else "train"


def cap_records(records):
    pool = [record for record in records if record["split_lock"] != "test"]
    rng = random.Random(17)
    selected = {}
    for name in CLASS_NAMES:
        keys = [record["key"] for record in pool if any(box[0] == name for box in record["boxes"])]
        rng.shuffle(keys)
        if len(keys) > IMAGES_PER_CLASS_CAP:
            keys = keys[:IMAGES_PER_CLASS_CAP]
        selected[name] = set(keys)
        print(f"cap {name}: {len(selected[name])} images (max {IMAGES_PER_CLASS_CAP})")
    prepared = []
    for record in records:
        if record["split_lock"] == "test":
            kept = list(record["boxes"])
        else:
            kept = [box for box in record["boxes"] if record["key"] in selected.get(box[0], set())]
        if not kept:
            continue
        copied = dict(record)
        copied["boxes"] = kept
        copied["split"] = assign_split(record)
        prepared.append(copied)
    return prepared


def write_tree(root: Path, records) -> dict:
    if root.exists():
        shutil.rmtree(root)
    for split in ("train", "val", "test"):
        (root / "images" / split).mkdir(parents=True)
        (root / "labels" / split).mkdir(parents=True)
    counts = {split: 0 for split in ("train", "val", "test")}
    box_counts = {name: 0 for name in CLASS_NAMES}
    manifest = ["source,item_code,split,image"]
    for index, record in enumerate(records):
        split = record["split"]
        suffix = record["path"].suffix.lower() or ".jpg"
        image_name = f"{split}_{index:05d}{suffix}"
        shutil.copy2(record["path"], root / "images" / split / image_name)
        lines = []
        for item_code, cx, cy, w, h, source_name in record["boxes"]:
            lines.append(
                f"{class_index(item_code)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
            )
            if split != "test":
                box_counts[item_code] += 1
            manifest.append(f"{source_name},{item_code},{split},{image_name}")
        (root / "labels" / split / f"{split}_{index:05d}.txt").write_text(
            "\n".join(lines) + "\n",
            encoding="utf-8",
        )
        counts[split] += 1
    # Ultralytics needs a val split. If the hash put everything in train, move one image.
    if counts["train"] > 1 and counts["val"] == 0:
        image = next((root / "images" / "train").iterdir())
        label = root / "labels" / "train" / (image.stem + ".txt")
        image.rename(root / "images" / "val" / image.name)
        label.rename(root / "labels" / "val" / label.name)
        counts["train"] -= 1
        counts["val"] += 1
        print("Moved one training image to val so the split is non-empty.")
    (root / "manifest.csv").write_text("\n".join(manifest) + "\n", encoding="utf-8")
    return {"images": counts, "boxes": box_counts}


def write_yaml(path: Path, dataset_root: Path, include_test: bool) -> None:
    payload = {
        "path": str(dataset_root),
        "train": "images/train",
        "val": "images/val",
        "names": {index: name for index, name in enumerate(CLASS_NAMES)},
    }
    if include_test:
        payload["test"] = "images/test"
    header = (
        "# HydroFlow V2 classes match ITEM_TYPES.\n"
        "# Cap is 1200 images per class (guidance about 1000-1500).\n"
        "# SKU-110K and DeepFashion2 are not the training set.\n"
    )
    path.write_text(header + yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")


prepared = cap_records(RECORDS)
public_records = [
    record for record in prepared if record["origin"] == "public" and record["split"] != "test"
]
full_records = list(prepared)
public_stats = write_tree(WORK / "public", public_records)
full_stats = write_tree(WORK / "full", full_records)
public_yaml = WORK / "public" / "data.yaml"
full_yaml = WORK / "full" / "data.yaml"
write_yaml(public_yaml, WORK / "public", include_test=False)
write_yaml(full_yaml, WORK / "full", include_test=full_stats["images"]["test"] > 0)

print("Public images", public_stats["images"])
print("Full images", full_stats["images"])
for name in CLASS_NAMES:
    boxes = full_stats["boxes"][name]
    note = ""
    if boxes < MIN_BOXES_BEFORE_MANUAL:
        note = " -> keep this class manual until it has at least 200 boxes"
    if name in LOCAL_REQUIRED:
        note += " -> local photos required"
    print(f"boxes {name}: {boxes}{note}")

if public_stats["images"]["train"] == 0 and full_stats["images"]["train"] == 0:
    raise SystemExit("No training images. Download Open Images or add local photos before training.")

status = {
    "public_yaml": str(public_yaml),
    "full_yaml": str(full_yaml),
    "public_train": public_stats["images"]["train"],
    "full_train": full_stats["images"]["train"],
    "test_images": full_stats["images"]["test"],
    "box_counts": full_stats["boxes"],
}
(WORK / "build_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
print("Wrote", public_yaml)
print("Wrote", full_yaml)
print(LOCAL_NOTE)


## Augmentation and training

Phase 1 freezes the backbone for 10 epochs on the public subset, starting from COCO pretrained `yolo11s.pt`.

Phase 2 fine-tunes every layer for 80 epochs at a lower learning rate on public plus local images. Image size is 640. Early stopping patience is 20. Augmentations are HSV, mosaic, mixup, copy-paste, scale, and blur.

The blur chance is raised through Ultralytics' Albumentations hook. HSV, mosaic, mixup, copy-paste, and scale are the `train` arguments below.


In [ ]:
def enable_blur_augmentation() -> None:
    import albumentations as A
    from ultralytics.data.augment import Albumentations

    stronger = A.Compose(
        [
            A.Blur(p=0.2, blur_limit=7),
            A.MedianBlur(p=0.1, blur_limit=5),
            A.CLAHE(p=0.05),
        ]
    )
    original_init = Albumentations.__init__

    def patched_init(self, p=1.0, transforms=None):
        try:
            original_init(self, p=p)
        except TypeError:
            original_init(self, p=p, transforms=transforms)
        self.transform = stronger
        self.p = p

    Albumentations.__init__ = patched_init
    print("Blur augmentation enabled with HSV, mosaic, mixup, copy-paste, and scale.")


TRAIN_AUGMENT = dict(
    imgsz=640,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    scale=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    close_mosaic=10,
    seed=17,
)
enable_blur_augmentation()
print(TRAIN_AUGMENT)


In [ ]:
import torch
from ultralytics import YOLO

status = json.loads((WORK / "build_status.json").read_text(encoding="utf-8"))
device = 0 if torch.cuda.is_available() else "cpu"
print("Training device:", device)

if status["public_train"] > 0:
    frozen = YOLO("yolo11s.pt")
    frozen_result = frozen.train(
        data=status["public_yaml"],
        epochs=10,
        freeze=10,
        batch=16,
        device=device,
        project=str(RUNS),
        name="yolo11s_public_frozen",
        exist_ok=True,
        pretrained=True,
        **TRAIN_AUGMENT,
    )
    phase1_best = Path(frozen_result.save_dir) / "weights" / "best.pt"
else:
    print("No public images. Phase 2 will start directly from yolo11s.pt.")
    phase1_best = None

weights = str(phase1_best) if phase1_best and phase1_best.is_file() else "yolo11s.pt"
model = YOLO(weights)
phase2_augment = dict(TRAIN_AUGMENT)
print("Phase 2: YOLO11s from COCO weights, imgsz 640, epochs 80, patience 20")
fine_tune = model.train(
    data=status["full_yaml"],
    epochs=80,
    patience=phase2_augment.pop("patience"),
    imgsz=phase2_augment.pop("imgsz"),
    freeze=0,
    lr0=0.001,
    lrf=0.01,
    batch=16,
    device=device,
    project=str(RUNS),
    name="yolo11s_v2",
    exist_ok=True,
    **phase2_augment,
)
best = Path(fine_tune.save_dir) / "weights" / "best.pt"
if not best.is_file():
    raise FileNotFoundError(best)
status["phase2_best"] = str(best)
(WORK / "build_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
print("Phase 2 best.pt:", best)


## Metrics

Report mAP50, mAP50-95, and per-class precision, recall, and AP. When held-out local sessions exist, those images are the split that matters. A class under 200 train/val boxes is called out again so it can stay manual.


In [ ]:
from ultralytics import YOLO

status = json.loads((WORK / "build_status.json").read_text(encoding="utf-8"))
best = Path(status["phase2_best"])
model = YOLO(str(best))
split = "test" if status.get("test_images", 0) else "val"
metrics = model.val(data=status["full_yaml"], imgsz=640, split=split)
box = metrics.box
print("split", split)
print("mAP50", float(box.map50))
print("mAP50-95", float(box.map))
print("precision", float(box.mp))
print("recall", float(box.mr))
names = metrics.names or {index: name for index, name in enumerate(CLASS_NAMES)}
per_class_ap = list(box.maps) if box.maps is not None else []
precisions = list(getattr(box, "p", []) or [])
recalls = list(getattr(box, "r", []) or [])
for index, ap in enumerate(per_class_ap):
    label = names.get(index, CLASS_NAMES[index] if index < len(CLASS_NAMES) else str(index))
    precision = float(precisions[index]) if index < len(precisions) else float("nan")
    recall = float(recalls[index]) if index < len(recalls) else float("nan")
    boxes = int(status.get("box_counts", {}).get(str(label), status.get("box_counts", {}).get(label, 0)))
    manual = " manual-until-200-boxes" if boxes < MIN_BOXES_BEFORE_MANUAL else ""
    print(f"{label} AP50-95={float(ap):.4f} P={precision:.4f} R={recall:.4f} boxes={boxes}{manual}")


## Save `best.pt`

The file is copied to `ml/models/object_detect/v2/best.pt` inside a Drive copy of HydroFlow V2 when that folder is mounted, and a browser download starts so you can place the same file in the working tree. The app uses this single model when the file exists. Otherwise it keeps the three YOLOv8n category weights.


In [ ]:
status = json.loads((WORK / "build_status.json").read_text(encoding="utf-8"))
best = Path(status["phase2_best"])
if not best.is_file():
    raise FileNotFoundError(best)

destinations = []
drive_repo = DRIVE / "HydroFlow_V2"
if (drive_repo / "backend").is_dir() and (drive_repo / "ml").is_dir():
    destinations.append(drive_repo / "ml" / "models" / "object_detect" / "v2" / "best.pt")
cwd = Path.cwd().resolve()
for parent in [cwd, *cwd.parents]:
    if (parent / "backend").is_dir() and (parent / "ml").is_dir():
        destinations.append(parent / "ml" / "models" / "object_detect" / "v2" / "best.pt")
        break
destinations.append(Path("/content/hydroflow_detect/best.pt"))

saved = []
for dest in destinations:
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, dest)
    saved.append(dest)
    print("Saved", dest)

try:
    from google.colab import files

    files.download(str(best))
    print("Browser download started. Save it as ml/models/object_detect/v2/best.pt")
except Exception as exc:
    print("Browser download was not started:", exc)

print("App weights path: ml/models/object_detect/v2/best.pt")
print("If that file is absent, the app keeps the three YOLOv8n category weights.")
print(LOCAL_NOTE)
for path in saved:
    print(path, path.stat().st_size, "bytes")
